# M1 output validation notebook (Ganesh)

Checks **every cleansed file M1 produced**, cell by cell, against the raw data in `data/`:

| File | What is checked |
|---|---|
| `outputs/complaints_clean.parquet` | contract, every column of every row vs raw CSV, ward recovery recomputed independently, work-order links, issue ids, dedup metrics recomputed |
| `outputs/wo_trail.parquet` | every cell recomputed from raw work orders and complaints |
| `outputs/open_issues.parquet` | every cell recomputed from `complaints_clean` and raw data, road/ward consistency |
| `outputs/dedup_metrics.json` | keys, ranges, and the numbers recomputed from the parquet |

**How to read it:** each check prints `PASS`, `FAIL` or `INFO`. Nothing is hidden or tuned to pass.
`INFO` lines are measurements with no fixed pass mark (for example how often the cleaned category agrees with the work order).
The raw CSVs are read directly here, **not** through the pipeline loaders, so the checks are independent of the code being tested.

Run all cells top to bottom (`Run All`). Regenerate the outputs first with `python -c "from pipeline.connect import run; run('data','outputs')"` if you changed the pipeline.

## 0. Setup and load everything

In [10]:
import json, os, re, sys
from itertools import combinations
from pathlib import Path
import numpy as np
import pandas as pd
from shapely.geometry import Point, shape

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 60)

def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "contracts" / "schemas.py").exists():
            return q
    raise FileNotFoundError("run this notebook inside the repo")

ROOT = find_root(); sys.path.insert(0, str(ROOT))
OUT = Path(os.environ.get("M1_OUT", ROOT / "outputs"))   # set env M1_OUT to test another folder
DATA = ROOT / "data"
print("repo:", ROOT); print("outputs:", OUT)

cc = pd.read_parquet(OUT / "complaints_clean.parquet")
tr = pd.read_parquet(OUT / "wo_trail.parquet")
oi = pd.read_parquet(OUT / "open_issues.parquet")
with open(OUT / "dedup_metrics.json", encoding="utf-8") as f:
    metrics = json.load(f)

# raw data, read directly (independent of pipeline code)
raw_c = pd.read_csv(DATA / "complaints.csv", encoding="utf-8")
raw_c["received_at"] = pd.to_datetime(raw_c.received_at, format="%Y-%m-%d %H:%M")
raw_wo = pd.read_csv(DATA / "work_orders.csv", encoding="utf-8")
raw_wo["completed_at"] = pd.to_datetime(raw_wo.completed_at, format="%Y-%m-%d %H:%M")
wards = pd.read_csv(DATA / "wards.csv", encoding="utf-8")
roads = pd.read_csv(DATA / "roads.csv", encoding="utf-8")
with open(DATA / "wards.geojson", encoding="utf-8") as f:
    geo = json.load(f)

for name, df in (("complaints_clean", cc), ("wo_trail", tr), ("open_issues", oi)):
    print(f"{name:18s} rows={len(df):6d} cols={df.shape[1]}")

repo: D:\CIL\Impact-lab-team-6
outputs: D:\CIL\Impact-lab-team-6\outputs
complaints_clean   rows=  6501 cols=14
wo_trail           rows=  2499 cols=6
open_issues        rows=   687 cols=8


In [11]:
RESULTS = []
def _add(section, name, status, detail):
    RESULTS.append(dict(section=section, check=name, status=status, detail=str(detail)))
    print(f"{status:5s} {name}" + (f"  | {detail}" if str(detail) else ""))
def check(section, name, ok, detail=""): _add(section, name, "PASS" if bool(ok) else "FAIL", detail)
def info(section, name, detail=""): _add(section, name, "INFO", detail)

def same(a: pd.Series, b: pd.Series, tol=None):
    """Cell-by-cell equality, NaN == NaN, optional numeric tolerance. Returns boolean Series."""
    a, b = a.reset_index(drop=True), b.reset_index(drop=True)
    both_na = a.isna() & b.isna()
    if tol is not None:
        eq = (a.astype(float) - b.astype(float)).abs() <= tol
    else:
        eq = a == b
    return (eq.fillna(False) | both_na)

def compare(section, name, got, want, tol=None, key=None):
    ok = same(got, want, tol)
    bad = int((~ok).sum())
    detail = f"{len(ok) - bad}/{len(ok)} cells match"
    check(section, name, bad == 0, detail)
    if bad:
        k = key if key is not None else pd.Series(range(len(ok)))
        ex = pd.DataFrame({"key": k.reset_index(drop=True), "got": got.reset_index(drop=True),
                           "expected": want.reset_index(drop=True)})[~ok].head(5)
        display(ex)

## 1. Contract: columns, order and types (`contracts/schemas.py`)

In [12]:
from contracts.schemas import SCHEMAS, validate
S = "1 contract"
for name, df in (("complaints_clean", cc), ("wo_trail", tr), ("open_issues", oi)):
    check(S, f"{name}: exact column list and order", list(df.columns) == list(SCHEMAS[name]),
          "" if list(df.columns) == list(SCHEMAS[name]) else f"got {list(df.columns)}")
    probs = validate(df, name)
    check(S, f"{name}: contracts.schemas.validate() has no problems", probs == [], probs)
dtypes = pd.DataFrame({n: df.dtypes.astype(str) for n, df in (("complaints_clean", cc), ("wo_trail", tr), ("open_issues", oi))})
display(dtypes.fillna(""))

PASS  complaints_clean: exact column list and order
PASS  complaints_clean: contracts.schemas.validate() has no problems  | []
PASS  wo_trail: exact column list and order
PASS  wo_trail: contracts.schemas.validate() has no problems  | []
PASS  open_issues: exact column list and order
PASS  open_issues: contracts.schemas.validate() has no problems  | []


,complaints_clean,wo_trail,open_issues
category_clean,str,,str
channel,str,,
channels,,str,
complaint_id,str,,
english_summary,str,,
first_complaint_at,,datetime64[us],
first_reported_at,,,datetime64[us]
hours_to_complete,,float64,
issue_id,str,,str
language,str,,


## 2. `complaints_clean.parquet`: row identity and every raw column, cell by cell

In [13]:
S = "2 complaints_clean"
check(S, "complaint_id is unique", cc.complaint_id.is_unique)
check(S, "complaint_id has no nulls", cc.complaint_id.notna().all())
check(S, "same complaint ids as raw complaints.csv (none lost, none invented)",
      set(cc.complaint_id) == set(raw_c.complaint_id),
      f"clean={len(cc)} raw={len(raw_c)}")
check(S, "complaint_id format BMC25-nnnnnn", cc.complaint_id.str.fullmatch(r"BMC25-\d{6}").all())

m = cc.merge(raw_c, on="complaint_id", suffixes=("", "_raw"), how="inner")
check(S, "every clean row joins to exactly one raw row", len(m) == len(cc))
m = m.reset_index(drop=True)

compare(S, "received_at identical to raw", m.received_at, m.received_at_raw, key=m.complaint_id)
compare(S, "channel identical to raw", m.channel, m.channel_raw, key=m.complaint_id)
compare(S, "lat identical to raw (NaN where raw NaN)", m.lat, m.lat_raw, tol=1e-9, key=m.complaint_id)
compare(S, "lon identical to raw (NaN where raw NaN)", m.lon, m.lon_raw, tol=1e-9, key=m.complaint_id)
compare(S, "language identical to raw", m.language, m.language_raw, key=m.complaint_id)
compare(S, "status identical to raw", m.status, m.status_raw, key=m.complaint_id)

PASS  complaint_id is unique
PASS  complaint_id has no nulls
PASS  same complaint ids as raw complaints.csv (none lost, none invented)  | clean=6501 raw=6501
PASS  complaint_id format BMC25-nnnnnn
PASS  every clean row joins to exactly one raw row
PASS  received_at identical to raw  | 6501/6501 cells match
PASS  channel identical to raw  | 6501/6501 cells match
PASS  lat identical to raw (NaN where raw NaN)  | 6501/6501 cells match
PASS  lon identical to raw (NaN where raw NaN)  | 6501/6501 cells match
PASS  language identical to raw  | 6501/6501 cells match
PASS  status identical to raw  | 6501/6501 cells match


### 2b. Ward recovery, recomputed independently

`ward_source` must say how each ward was obtained: `given` (raw value), `gps` (point-in-polygon), `locality` (gazetteer).

In [14]:
S = "2b wards"
polys = [(f["properties"]["ward_code"], shape(f["geometry"])) for f in geo["features"]]
def locate(lat, lon):
    if pd.isna(lat) or pd.isna(lon): return None
    p = Point(lon, lat)
    return next((c for c, poly in polys if poly.covers(p)), None)

valid_wards = set(wards.ward_code)
check(S, "ward_code has no nulls", cc.ward_code.notna().all(), f"coverage {cc.ward_code.notna().mean():.2%}")
check(S, "every ward_code is a real ward (24 wards)", cc.ward_code.isin(valid_wards).all())
check(S, "ward_source only given/gps/locality", cc.ward_source.isin(["given", "gps", "locality"]).all(),
      cc.ward_source.value_counts().to_dict())

check(S, "ward_source=='given' exactly where raw ward is present",
      ((m.ward_source == "given") == m.ward_code_raw.notna()).all())
g = m[m.ward_source == "given"]
compare(S, "given rows: ward equals raw ward", g.ward_code, g.ward_code_raw, key=g.complaint_id)

r = m[m.ward_source == "gps"]
recomputed = r.apply(lambda x: locate(x.lat, x.lon), axis=1)
compare(S, "gps rows: ward equals independent point-in-polygon on wards.geojson", r.ward_code, recomputed, key=r.complaint_id)
check(S, "gps rows all have raw lat/lon and a blank raw ward", (r.lat_raw.notna() & r.ward_code_raw.isna()).all(), f"{len(r)} rows")

l = m[m.ward_source == "locality"]
no_gps_hit = l.apply(lambda x: locate(x.lat, x.lon), axis=1).isna()
check(S, "locality rows: GPS could not have resolved them (so locality was the right fallback)", no_gps_hit.all(), f"{len(l)} rows")
check(S, "locality rows had a blank raw ward", l.ward_code_raw.isna().all())

# agreement between given ward and GPS where both exist (sanity of the geojson/gazetteer)
both = m[m.ward_code_raw.notna() & m.lat_raw.notna()]
agree = (both.apply(lambda x: locate(x.lat_raw, x.lon_raw), axis=1) == both.ward_code_raw).mean()
info(S, "raw given ward vs GPS polygon agreement on rows having both", f"{agree:.2%} of {len(both)}")

PASS  ward_code has no nulls  | coverage 100.00%
PASS  every ward_code is a real ward (24 wards)
PASS  ward_source only given/gps/locality  | {'given': 6044, 'gps': 271, 'locality': 186}
PASS  ward_source=='given' exactly where raw ward is present
PASS  given rows: ward equals raw ward  | 6044/6044 cells match
PASS  gps rows: ward equals independent point-in-polygon on wards.geojson  | 271/271 cells match
PASS  gps rows all have raw lat/lon and a blank raw ward  | 271 rows
PASS  locality rows: GPS could not have resolved them (so locality was the right fallback)  | 186 rows
PASS  locality rows had a blank raw ward
INFO  raw given ward vs GPS polygon agreement on rows having both  | 99.71% of 3506


### 2c. Category, severity, summary

In [15]:
S = "2c extraction"
CATS = {"Pothole", "Waterlogging", "Open or broken manhole", "Choked drain / nullah overflow",
        "Road cave-in", "Broken paver blocks"}
check(S, "category_clean has no nulls", cc.category_clean.notna().all())
check(S, "category_clean only the six real categories (no 'Road related'/'Other' left)",
      cc.category_clean.isin(CATS).all(), cc.category_clean.value_counts().to_dict())
check(S, "severity_1_5 has no nulls", cc.severity_1_5.notna().all())
check(S, "severity_1_5 integer in 1..5", (cc.severity_1_5 == cc.severity_1_5.round()).all() and cc.severity_1_5.between(1, 5).all(),
      cc.severity_1_5.value_counts().sort_index().to_dict())
check(S, "english_summary non-empty text on every row",
      cc.english_summary.notna().all() and (cc.english_summary.str.strip().str.len() > 0).all())

mc = m[m.category_reported_raw.isin(CATS)] if "category_reported_raw" in m else m[m.category_reported.isin(CATS)]
rep = "category_reported_raw" if "category_reported_raw" in m else "category_reported"
info(S, "share of rows keeping the citizen's specific category", f"{(mc.category_clean == mc[rep]).mean():.2%} of {len(mc)}")
info(S, "extraction source counts (rules = keyword fallback, claude = LLM)", metrics.get("extraction_source_counts"))
if metrics.get("extraction_source_counts", {}).get("claude", 0) == 0:
    info(S, "NOTE", "Claude extraction has NOT run for any row; categories/severity come from the keyword fallback")

PASS  category_clean has no nulls
PASS  category_clean only the six real categories (no 'Road related'/'Other' left)  | {'Pothole': 2861, 'Waterlogging': 1414, 'Open or broken manhole': 719, 'Choked drain / nullah overflow': 593, 'Broken paver blocks': 457, 'Road cave-in': 457}
PASS  severity_1_5 has no nulls
PASS  severity_1_5 integer in 1..5  | {3: 4048, 4: 2179, 5: 274}
PASS  english_summary non-empty text on every row
INFO  share of rows keeping the citizen's specific category  | 100.00% of 6102
INFO  extraction source counts (rules = keyword fallback, claude = LLM)  | {'rules': 6501}
INFO  NOTE  | Claude extraction has NOT run for any row; categories/severity come from the keyword fallback


### 2d. Complaint to work-order links

In [16]:
S = "2d links"
raw_links = (raw_wo.dropna(subset=["linked_complaint_ids"])
             .assign(complaint_id=lambda d: d.linked_complaint_ids.str.split("|"))
             .explode("complaint_id")[["complaint_id", "work_order_id"]])
check(S, "no complaint is linked to two work orders in the raw data", raw_links.complaint_id.is_unique)
want = cc[["complaint_id"]].merge(raw_links, on="complaint_id", how="left").work_order_id
compare(S, "linked_work_order_id equals the work order that lists the complaint (all 6,501 cells)",
        cc.linked_work_order_id, want, key=cc.complaint_id)
check(S, "every linked work order exists in work_orders.csv",
      cc.linked_work_order_id.dropna().isin(raw_wo.work_order_id).all())
check(S, "number of linked complaints equals raw link count", cc.linked_work_order_id.notna().sum() == len(raw_links),
      f"{cc.linked_work_order_id.notna().sum()} vs {len(raw_links)}")

PASS  no complaint is linked to two work orders in the raw data
PASS  linked_work_order_id equals the work order that lists the complaint (all 6,501 cells)  | 6501/6501 cells match
PASS  every linked work order exists in work_orders.csv
PASS  number of linked complaints equals raw link count  | 4948 vs 4948


### 2e. Issue ids (de-duplication result)

In [17]:
S = "2e issues"
check(S, "issue_id has no nulls", cc.issue_id.notna().all())
check(S, "issue_id format ISS-nnnnn", cc.issue_id.str.fullmatch(r"ISS-\d{5}").all())
by = cc.groupby("issue_id")
check(S, "every issue lies in exactly one ward", (by.ward_code.nunique() == 1).all())
info(S, "distinct issues / complaints", f"{cc.issue_id.nunique()} issues from {len(cc)} complaints")
sizes = by.size()
info(S, "issue size distribution", sizes.value_counts().sort_index().head(8).to_dict())
info(S, "issues containing more than one cleaned category", f"{(by.category_clean.nunique() > 1).mean():.2%}")
check(S, "no issue larger than 10 complaints (sanity against giant merged clusters)", sizes.max() <= 10, f"max size {sizes.max()}")

# recompute the dedup metrics from the parquet and compare with dedup_metrics.json
def pairs(frame, col):
    out = set()
    for _, g in frame.groupby(col):
        out.update(combinations(sorted(g.complaint_id), 2))
    return out
linked = cc.dropna(subset=["linked_work_order_id"])
truth, pred = pairs(linked, "linked_work_order_id"), pairs(linked, "issue_id")
tp = len(truth & pred)
P_, R_ = tp / len(pred), tp / len(truth)
F_ = 2 * P_ * R_ / (P_ + R_)
print(f"recomputed: precision={P_:.4f} recall={R_:.4f} f1={F_:.4f}  (truth pairs={len(truth)}, predicted={len(pred)})")
S2 = "2f dedup metrics"
check(S2, "precision in JSON matches recomputation", abs(P_ - metrics["precision"]) < 1e-3, f"{metrics['precision']} vs {P_:.4f}")
check(S2, "recall in JSON matches recomputation", abs(R_ - metrics["recall"]) < 1e-3, f"{metrics['recall']} vs {R_:.4f}")
check(S2, "f1 in JSON matches recomputation", abs(F_ - metrics["f1"]) < 1e-3, f"{metrics['f1']} vs {F_:.4f}")
check(S2, "dedup F1 >= 0.60 (tests/test_connect.py bar)", F_ >= 0.60, f"{F_:.4f}")

PASS  issue_id has no nulls
PASS  issue_id format ISS-nnnnn
PASS  every issue lies in exactly one ward
INFO  distinct issues / complaints  | 3529 issues from 6501 complaints
INFO  issue size distribution  | {1: 1790, 2: 1025, 3: 417, 4: 155, 5: 88, 6: 38, 7: 9, 8: 5}
INFO  issues containing more than one cleaned category  | 9.72%
PASS  no issue larger than 10 complaints (sanity against giant merged clusters)  | max size 10
recomputed: precision=0.7131 recall=0.6843 f1=0.6984  (truth pairs=3868, predicted=3712)
PASS  precision in JSON matches recomputation  | 0.7131 vs 0.7131
PASS  recall in JSON matches recomputation  | 0.6843 vs 0.6843
PASS  f1 in JSON matches recomputation  | 0.6984 vs 0.6984
PASS  dedup F1 >= 0.60 (tests/test_connect.py bar)  | 0.6984


## 3. `wo_trail.parquet`: every cell recomputed from raw

In [18]:
S = "3 wo_trail"
check(S, "work_order_id unique, no nulls", tr.work_order_id.is_unique and tr.work_order_id.notna().all())
check(S, "same work orders as raw work_orders.csv", set(tr.work_order_id) == set(raw_wo.work_order_id), f"{len(tr)} rows vs {len(raw_wo)}")

rc = raw_c.set_index("complaint_id")
rows = []
for r in raw_wo.itertuples():
    ids = [x for x in str(r.linked_complaint_ids).split("|") if x] if pd.notna(r.linked_complaint_ids) else []
    g = rc.reindex(ids).dropna(subset=["received_at"])
    first = g.received_at.min() if len(g) else pd.NaT
    rows.append(dict(work_order_id=r.work_order_id, n_complaints=len(ids), first_complaint_at=first,
                     channels="|".join(sorted(g.channel.unique())),
                     total_upvotes=int(g.upvotes.sum()) if len(g) else 0,
                     hours_to_complete=(r.completed_at - first).total_seconds() / 3600 if pd.notna(first) else np.nan))
exp = pd.DataFrame(rows)
t = tr.merge(exp, on="work_order_id", suffixes=("", "_exp"), how="inner").reset_index(drop=True)
check(S, "every trail row joins to an expected row", len(t) == len(tr))
compare(S, "n_complaints", t.n_complaints, t.n_complaints_exp, key=t.work_order_id)
compare(S, "first_complaint_at", t.first_complaint_at, t.first_complaint_at_exp, key=t.work_order_id)
compare(S, "channels", t.channels, t.channels_exp, key=t.work_order_id)
compare(S, "total_upvotes", t.total_upvotes, t.total_upvotes_exp, key=t.work_order_id)
compare(S, "hours_to_complete (tolerance 1e-6 h)", t.hours_to_complete, t.hours_to_complete_exp, tol=1e-6, key=t.work_order_id)

check(S, "n_complaints is never negative", (tr.n_complaints >= 0).all())
check(S, "total_upvotes is never negative", (tr.total_upvotes >= 0).all())
check(S, "sum of n_complaints equals linked complaints in complaints_clean", tr.n_complaints.sum() == cc.linked_work_order_id.notna().sum(),
      f"{tr.n_complaints.sum()} vs {cc.linked_work_order_id.notna().sum()}")
check(S, "work orders with no linked complaint have null first_complaint_at and empty channels",
      ((tr.n_complaints == 0) == tr.first_complaint_at.isna()).all(), f"{(tr.n_complaints == 0).sum()} unlinked work orders")
neg = tr[tr.hours_to_complete < 0]
check(S, "hours_to_complete is never negative", len(neg) == 0, f"{len(neg)} negative rows")
if len(neg):
    display(neg.head())
info(S, "hours_to_complete summary", tr.hours_to_complete.describe().round(1).to_dict())

PASS  work_order_id unique, no nulls
PASS  same work orders as raw work_orders.csv  | 2499 rows vs 2499
PASS  every trail row joins to an expected row
PASS  n_complaints  | 2499/2499 cells match
PASS  first_complaint_at  | 2499/2499 cells match
PASS  channels  | 2499/2499 cells match
PASS  total_upvotes  | 2499/2499 cells match
PASS  hours_to_complete (tolerance 1e-6 h)  | 2499/2499 cells match
PASS  n_complaints is never negative
PASS  total_upvotes is never negative
PASS  sum of n_complaints equals linked complaints in complaints_clean  | 4948 vs 4948
PASS  work orders with no linked complaint have null first_complaint_at and empty channels  | 45 unlinked work orders
PASS  hours_to_complete is never negative  | 0 negative rows
INFO  hours_to_complete summary  | {'count': 2454.0, 'mean': 84.2, 'std': 55.7, 'min': 9.1, '25%': 48.3, '50%': 70.2, '75%': 108.1, 'max': 602.8}


## 4. `open_issues.parquet`: every cell recomputed

In [19]:
S = "4 open_issues"
OPEN = ["Open", "In Progress"]
check(S, "issue_id unique, no nulls", oi.issue_id.is_unique and oi.issue_id.notna().all())
check(S, "no nulls in any column", not oi.isna().any().any(), oi.isna().sum()[oi.isna().sum() > 0].to_dict())
n_open = int(raw_c.status.isin(OPEN).sum())
check(S, "0 < issues <= number of open complaints", 0 < len(oi) <= n_open, f"{len(oi)} issues from {n_open} open complaints")

open_ids = set(cc[cc.status.isin(OPEN)].issue_id)
check(S, "issues are exactly those containing at least one Open/In Progress complaint", set(oi.issue_id) == open_ids,
      f"{len(open_ids)} expected, {len(oi)} present")

mm = cc.merge(raw_c[["complaint_id", "upvotes"]], on="complaint_id", how="left")
grp = mm[mm.issue_id.isin(open_ids)].groupby("issue_id")
exp = pd.DataFrame({
    "n_complaints": grp.size(),
    "total_upvotes": grp.upvotes.sum(),
    "first_reported_at": grp.received_at.min(),
    "severity_1_5": grp.severity_1_5.max(),
    "category_clean": grp.category_clean.agg(lambda s: s.mode().iloc[0]),
    "ward_code": grp.ward_code.agg(lambda s: s.dropna().mode().iloc[0]),
}).reset_index()
o = oi.merge(exp, on="issue_id", suffixes=("", "_exp"), how="inner").reset_index(drop=True)
check(S, "every open issue joins to an expected row", len(o) == len(oi))
for col in ("n_complaints", "total_upvotes", "first_reported_at", "severity_1_5", "category_clean", "ward_code"):
    compare(S, f"{col} recomputed from complaints_clean", o[col], o[col + "_exp"], key=o.issue_id)

check(S, "ward_code is a real ward", oi.ward_code.isin(valid_wards).all())
check(S, "road_id exists in roads.csv", oi.road_id.isin(roads.road_id).all())
rw = oi.merge(roads[["road_id", "ward_code"]], on="road_id", suffixes=("", "_road"))
check(S, "road is in the same ward as the issue", (rw.ward_code == rw.ward_code_road).all(), f"{(rw.ward_code != rw.ward_code_road).sum()} mismatches")
check(S, "category_clean only the six real categories", oi.category_clean.isin(CATS).all())
check(S, "severity_1_5 in 1..5", oi.severity_1_5.between(1, 5).all())
check(S, "n_complaints >= 1 everywhere", (oi.n_complaints >= 1).all())
check(S, "total n_complaints equals complaints inside those issues", oi.n_complaints.sum() == cc.issue_id.isin(open_ids).sum(),
      f"{oi.n_complaints.sum()} vs {cc.issue_id.isin(open_ids).sum()}")
info(S, "issues per ward (top 5)", oi.ward_code.value_counts().head(5).to_dict())
info(S, "roads that receive at least one open issue", f"{oi.road_id.nunique()} of {len(roads)}")

PASS  issue_id unique, no nulls
PASS  no nulls in any column  | {}
PASS  0 < issues <= number of open complaints  | 687 issues from 896 open complaints
PASS  issues are exactly those containing at least one Open/In Progress complaint  | 687 expected, 687 present
PASS  every open issue joins to an expected row
PASS  n_complaints recomputed from complaints_clean  | 687/687 cells match
PASS  total_upvotes recomputed from complaints_clean  | 687/687 cells match
PASS  first_reported_at recomputed from complaints_clean  | 687/687 cells match
PASS  severity_1_5 recomputed from complaints_clean  | 687/687 cells match
PASS  category_clean recomputed from complaints_clean  | 687/687 cells match
PASS  ward_code recomputed from complaints_clean  | 687/687 cells match
PASS  ward_code is a real ward
PASS  road_id exists in roads.csv
PASS  road is in the same ward as the issue  | 0 mismatches
PASS  category_clean only the six real categories
PASS  severity_1_5 in 1..5
PASS  n_complaints >= 1 everywhe

## 5. Null scan: every column of every file

In [20]:
S = "5 nulls"
allowed = {  # column -> why nulls are legitimate
    "complaints_clean": {"lat": "raw GPS missing", "lon": "raw GPS missing", "linked_work_order_id": "complaint not linked to any work order"},
    "wo_trail": {"first_complaint_at": "work order has no linked complaint", "hours_to_complete": "work order has no linked complaint"},
    "open_issues": {},
}
rows = []
for name, df in (("complaints_clean", cc), ("wo_trail", tr), ("open_issues", oi)):
    for col in df.columns:
        n = int(df[col].isna().sum())
        ok = n == 0 or col in allowed[name]
        rows.append(dict(file=name, column=col, nulls=n, pct=f"{n/len(df):.1%}", allowed_reason=allowed[name].get(col, ""), status="PASS" if ok else "FAIL"))
        RESULTS.append(dict(section=S, check=f"{name}.{col} nulls only where legitimate", status="PASS" if ok else "FAIL", detail=f"{n} nulls"))
nulls = pd.DataFrame(rows)
display(nulls)
print("null-scan FAILs:", int((nulls.status == "FAIL").sum()))

# null lat/lon in the clean file must be exactly the raw nulls
check(S, "null lat/lon in complaints_clean equals null lat/lon in raw", (cc.lat.isna().sum() == raw_c.lat.isna().sum()) and (cc.lon.isna().sum() == raw_c.lon.isna().sum()),
      f"{cc.lat.isna().sum()} vs {raw_c.lat.isna().sum()}")

,file,column,nulls,pct,allowed_reason,status
0,complaints_clean,complaint_id,0,0.0%,,PASS
1,complaints_clean,received_at,0,0.0%,,PASS
2,complaints_clean,channel,0,0.0%,,PASS
3,complaints_clean,ward_code,0,0.0%,,PASS
4,complaints_clean,ward_source,0,0.0%,,PASS
5,complaints_clean,lat,2724,41.9%,raw GPS missing,PASS
6,complaints_clean,lon,2724,41.9%,raw GPS missing,PASS
7,complaints_clean,category_clean,0,0.0%,,PASS
8,complaints_clean,severity_1_5,0,0.0%,,PASS
9,complaints_clean,language,0,0.0%,,PASS


null-scan FAILs: 0
PASS  null lat/lon in complaints_clean equals null lat/lon in raw  | 2724 vs 2724


## 6. `dedup_metrics.json`

In [21]:
S = "6 json"
need = {"precision", "recall", "f1", "n_truth_pairs", "n_pred_pairs", "extraction_source_counts", "ward_source_counts"}
check(S, "all expected keys present", need <= set(metrics), sorted(need - set(metrics)))
check(S, "precision/recall/f1 are numbers in [0, 1]", all(isinstance(metrics[k], (int, float)) and 0 <= metrics[k] <= 1 for k in ("precision", "recall", "f1")))
check(S, "n_truth_pairs equals recomputed", metrics["n_truth_pairs"] == len(truth), f"{metrics['n_truth_pairs']} vs {len(truth)}")
check(S, "n_pred_pairs equals recomputed", metrics["n_pred_pairs"] == len(pred), f"{metrics['n_pred_pairs']} vs {len(pred)}")
check(S, "extraction_source_counts adds up to all complaints", sum(metrics["extraction_source_counts"].values()) == len(cc), metrics["extraction_source_counts"])
vc = cc.ward_source.value_counts().to_dict()
check(S, "ward_source_counts equals counts in complaints_clean", {k: v for k, v in metrics["ward_source_counts"].items() if k != "unresolved"} == vc,
      f"{metrics['ward_source_counts']} vs {vc}")
display(pd.Series(metrics, name="dedup_metrics.json").to_frame())

PASS  all expected keys present  | []
PASS  precision/recall/f1 are numbers in [0, 1]
PASS  n_truth_pairs equals recomputed  | 3868 vs 3868
PASS  n_pred_pairs equals recomputed  | 3712 vs 3712
PASS  extraction_source_counts adds up to all complaints  | {'rules': 6501}
PASS  ward_source_counts equals counts in complaints_clean  | {'given': 6044, 'gps': 271, 'locality': 186} vs {'given': 6044, 'gps': 271, 'locality': 186}


,dedup_metrics.json
precision,0.7131
recall,0.6843
f1,0.6984
n_truth_pairs,3868
n_pred_pairs,3712
extraction_source_counts,{'rules': 6501}
ward_source_counts,"{'given': 6044, 'gps': 271, 'locality': 186}"


## 7. Informational quality measures (no pass mark)

These compare the cleaned values with the work order that fixed the complaint. They show how good the cleaning is. Several are weak while Claude extraction has not run; they are **not** turned into PASS/FAIL so they cannot hide a problem.

In [22]:
S = "7 quality"
wo_info = raw_wo[["work_order_id", "work_type", "ward_code", "sla_hours"]].rename(columns={"ward_code": "wo_ward", "work_type": "wo_type"})
q = cc.dropna(subset=["linked_work_order_id"]).merge(wo_info, left_on="linked_work_order_id", right_on="work_order_id")
sev_map = {24: 5, 48: 4, 72: 3, 120: 2, 168: 1}
q["sev_truth"] = q.sla_hours.map(sev_map)
info(S, "category_clean equals work-order work_type", f"{(q.category_clean == q.wo_type).mean():.1%} of {len(q)} linked complaints")
rep_col = "category_reported"
qr = q.merge(raw_c[["complaint_id", rep_col]], on="complaint_id")
wrong = qr[qr[rep_col] != qr.wo_type]
info(S, "...of complaints where the citizen's category was wrong, cleaned category correct", f"{(wrong.category_clean == wrong.wo_type).mean():.1%} of {len(wrong)}")
info(S, "severity_1_5 equals severity implied by work-order SLA hours", f"{(q.severity_1_5 == q.sev_truth).mean():.1%}")
info(S, "severity within +-1 of that truth", f"{((q.severity_1_5 - q.sev_truth).abs() <= 1).mean():.1%}")
info(S, "ward_code equals the work order's ward", f"{(q.ward_code == q.wo_ward).mean():.2%}")
rec = q[q.ward_source != "given"]
info(S, "recovered wards (gps/locality) equal the work order's ward", f"{(rec.ward_code == rec.wo_ward).mean():.2%} of {len(rec)}")
for src in ("gps", "locality"):
    r_ = rec[rec.ward_source == src]
    info(S, f"   by source: {src}", f"{(r_.ward_code == r_.wo_ward).mean():.2%} of {len(r_)}")

INFO  category_clean equals work-order work_type  | 87.3% of 4948 linked complaints
INFO  ...of complaints where the citizen's category was wrong, cleaned category correct  | 23.3% of 818
INFO  severity_1_5 equals severity implied by work-order SLA hours  | 26.0%
INFO  severity within +-1 of that truth  | 73.6%
INFO  ward_code equals the work order's ward  | 99.98%
INFO  recovered wards (gps/locality) equal the work order's ward  | 99.71% of 342
INFO     by source: gps  | 100.00% of 191
INFO     by source: locality  | 99.34% of 151


## 8. Final summary

In [23]:
res = pd.DataFrame(RESULTS)
summary = res.groupby(["section", "status"]).size().unstack(fill_value=0)
for c in ("PASS", "FAIL", "INFO"):
    if c not in summary: summary[c] = 0
display(summary[["PASS", "FAIL", "INFO"]])
n_fail = int((res.status == "FAIL").sum())
print(f"TOTAL  PASS={int((res.status=='PASS').sum())}  FAIL={n_fail}  INFO={int((res.status=='INFO').sum())}")
if n_fail:
    print("\nFAILED CHECKS:")
    display(res[res.status == "FAIL"][["section", "check", "detail"]])
else:
    print("\nAll checks passed.")

status,PASS,FAIL,INFO
section,,,
1 contract,6,0,0
2 complaints_clean,11,0,0
2b wards,9,0,1
2c extraction,5,0,3
2d links,4,0,0
2e issues,4,0,3
2f dedup metrics,4,0,0
3 wo_trail,13,0,1
4 open_issues,18,0,2


TOTAL  PASS=109  FAIL=0  INFO=18

All checks passed.


### Reading the result

- **FAIL** means a real mismatch between an output cell and what the raw data implies. Fix the pipeline, regenerate `outputs/`, rerun.
- **INFO** lines are honest quality measurements, not tests. Category and severity quality are limited until the Claude extraction has been run with an API key.
- The dedup precision/recall is **held-out** (each ward scored by a model that did not train on it), but the 0.30 cut-off was chosen while looking at this same data, so it is slightly optimistic.